In [1]:
!pip install beautifulsoup4

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/115.0'
]

def get_article_links(category_url, num_pages=5):
    """ BƯỚC 1: CÀO CHIỀU RỘNG (Đã sửa lỗi phân trang và lỗi thẻ) """
    all_articles = []

    for page in range(1, num_pages + 1):
        # SỬA LỖI 1: Trang 1 dùng URL gốc, từ trang 2 trở đi mới thêm -p
        if page == 1:
            page_url = category_url
        else:
            page_url = f"{category_url}-p{page}"

        print(f"[Chiều rộng] Đang quét: {page_url}")

        headers = {'User-Agent': random.choice(USER_AGENTS)}
        try:
            response = requests.get(page_url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"-> Dừng danh mục tại trang {page} (Mã lỗi: {response.status_code})")
                break

            soup = BeautifulSoup(response.text, 'html.parser')

            # SỬA LỖI 2: Tìm tất cả phần tử có class 'title-news' (chấp nhận cả h2, h3, div...)
            articles = soup.find_all(class_='title-news')

            if not articles:
                print(f"-> Trang {page} không chứa thẻ bài viết nào.")
                break

            for article in articles:
                a_tag = article.find('a')
                if a_tag and a_tag.get('href'):
                    title = a_tag.text.strip()
                    link = a_tag.get('href')

                    # Chuẩn hóa link (đôi khi link trả về dạng tương đối /tin-tuc... thì cần nối thêm domain)
                    if link.startswith('/'):
                        link = 'https://vnexpress.net' + link

                    if link not in [item['Đường dẫn'] for item in all_articles]:
                        all_articles.append({'Tiêu đề': title, 'Đường dẫn': link})

            time.sleep(random.uniform(0.5, 1.5))

        except Exception as e:
            print(f"Lỗi quét trang danh mục {page}: {e}")
            break

    return all_articles


def get_article_content(article_url):
    """ BƯỚC 2: CÀO CHIỀU SÂU """
    headers = {'User-Agent': random.choice(USER_AGENTS)}
    try:
        response = requests.get(article_url, headers=headers, timeout=10)
        if response.status_code != 200:
            return ""
        soup = BeautifulSoup(response.text, 'html.parser')
        paragraphs = soup.find_all('p', class_='Normal')
        return " ".join([p.text.strip() for p in paragraphs])
    except:
        return ""


if __name__ == "__main__":
    # Danh mục các trang con của Kinh doanh
    CATEGORIES = [
        "https://vnexpress.net/kinh-doanh/doanh-nghiep",
        "https://vnexpress.net/kinh-doanh/chung-khoan",
        "https://vnexpress.net/kinh-doanh/vi-mo",
        "https://vnexpress.net/kinh-doanh/ebank"
    ]
    PAGES_PER_CATEGORY = 50  # Thử nghiệm trước với 5 trang mỗi mục, chạy tốt bạn nâng lên 20-50 trang nhé
    OUTPUT_FILE = 'kho_du_lieu_nlp.csv'

    if not os.path.exists(OUTPUT_FILE):
        df_empty = pd.DataFrame(columns=['Tiêu đề', 'Đường dẫn', 'Nội dung chi tiết'])
        df_empty.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    all_scraped_links = []
    print("=== BẮT ĐẦU THU THẬP LINK ĐA DANH MỤC ===")
    for cat in CATEGORIES:
        print(f"\n>>> Quét danh mục: {cat}")
        links = get_article_links(cat, num_pages=PAGES_PER_CATEGORY)
        all_scraped_links.extend(links)

    total_links = len(all_scraped_links)
    print(f"\n Tổng số link thu thập được: {total_links} link.")

    if total_links > 0:
        print("\n=== BẮT ĐẦU CÀO CHIỀU SÂU & LƯU CUỐN CHIẾU ===")
        success_count = 0

        for index, article in enumerate(all_scraped_links):
            url = article['Đường dẫn']
            title = article['Tiêu đề']

            print(f"[{index + 1}/{total_links}] Đang cào sâu: {title[:40]}...")
            content = get_article_content(url)

            if content and len(content) > 150:
                df_row = pd.DataFrame([{
                    'Tiêu đề': title,
                    'Đường dẫn': url,
                    'Nội dung chi tiết': content
                }])
                df_row.to_csv(OUTPUT_FILE, mode='a', index=False, header=False, encoding='utf-8-sig')
                success_count += 1

            time.sleep(random.uniform(1.0, 2.0))

        print("\n" + "="*40)
        print(f" HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH!")
        print(f" Dữ liệu an toàn tại file: '{OUTPUT_FILE}'")
        print(f" Số bài viết đã tải xuống thành công: {success_count}/{total_links}")
        print("="*40)
    else:
        print("\nVẫn không tìm thấy link nào. Hãy kiểm tra lại kết nối mạng hoặc cấu trúc class của báo.")

=== BẮT ĐẦU THU THẬP LINK ĐA DANH MỤC ===

>>> Quét danh mục: https://vnexpress.net/kinh-doanh/doanh-nghiep
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p2
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p3
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p4
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p5
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p6
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p7
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p8
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p9
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p10
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p11
[Chiều rộng] Đang quét: https://vnexpress.net/kinh-doanh/doanh-nghiep-p12
[Chiều rộng] Đan